<a href="https://colab.research.google.com/github/YardenGoraly/Mujoco_fun/blob/main/MuJoCo_fun.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Initial setup, you shouldn't have to modify this code

# %pip install mujoco
# %pip install mediapy

import platform
import os
import subprocess
import mediapy as media

# Detect the operating system and configure GPU rendering accordingly.
if platform.system() == "Linux":
    # Assume Nvidia GPU is present.
    if subprocess.run("nvidia-smi", shell=True).returncode != 0:
        raise RuntimeError(
            "Cannot communicate with GPU. Make sure you are using a GPU runtime."
        )

    # Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
    NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
            f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")
    print("Setting environment variable for Nvidia GPU rendering (EGL).")
    os.environ["MUJOCO_GL"] = "egl"

elif platform.system() == "Darwin":
    # Assume running on macOS (Apple Silicon).
    print("Running on macOS. Setting environment variable for GPU rendering using GLFW.")
    os.environ["MUJOCO_GL"] = "glfw"

    media.set_ffmpeg("/opt/homebrew/bin/ffmpeg")
else:
    print("Unsupported platform. GPU rendering might not be configured correctly.")

# Check if MuJoCo installation was successful.
try:
    import mujoco as mj
    mj.MjModel.from_xml_string("<mujoco/>")
except Exception as e:
    raise RuntimeError(
        "Something went wrong during MuJoCo installation. Check the shell output above for more information."
    ) from e

print("MuJoCo installation successful.")

# Other imports and helper functions.
import time
import itertools
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=100)

# Graphics and plotting.
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# On Linux, ensure ffmpeg is installed (this is not applicable on macOS).
if platform.system() == "Linux":
    !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)

%pip install -q mediapy
import mediapy as media

from IPython.display import clear_output
clear_output()


In [ ]:
#More setup 
# %pip install robot_descriptions
# %pip install dm_control

from robot_descriptions import panda_mj_description
from IPython.display import HTML
from robot_descriptions.loaders.mujoco import load_robot_description
from dm_control import mjcf
import dm_control
import PIL.Image


def display_video(frames, framerate=30):
    height, width, _ = frames[0].shape
    dpi = 70
    orig_backend = matplotlib.get_backend()
    matplotlib.use('Agg')  # Switch to headless 'Agg' to inhibit figure rendering.
    fig, ax = plt.subplots(1, 1, figsize=(width / dpi, height / dpi), dpi=dpi)
    matplotlib.use(orig_backend)  # Switch back to the original backend.
    ax.set_axis_off()
    ax.set_aspect('equal')
    ax.set_position([0, 0, 1, 1])
    im = ax.imshow(frames[0])
    def update(frame):
      im.set_data(frame)
      return [im]
    interval = 1000/framerate
    anim = animation.FuncAnimation(fig=fig, func=update, frames=frames,
                                   interval=interval, blit=True, repeat=False)
    return HTML(anim.to_html5_video())


In [ ]:
# Get XML for Sawyer, hand, and ball
ball_xml = """
<mujoco model="ball">
    <worldbody>
        <body name="ball_body" pos="1.0 -0.2 0.95">
            <geom name="ball_geom" mass="0.01" friction="1.5" type="sphere" size="0.05" rgba="1 0 0 1"
                  solref="0.06 1" solimp="0.9 0.95 0.003 0.5 2"/>
        </body>
    </worldbody>
</mujoco>
"""

table_xml = """
<mujoco model="table">
    <worldbody>
        <body name="table_body" pos="1.0 -0.2 0.45">
            <geom name="table_geom" mass="200000" friction="0.8" type="box" solref="0.01 0.5" size="0.3 0.6 0.45" rgba="0.798 0.71 0.469 1"/>
        </body>
    </worldbody>
</mujoco>
"""

hand_path = "mujoco_menagerie/wonik_allegro/right_hand.xml"
sawyer_path = "mujoco_menagerie/rethink_robotics_sawyer/sawyer.xml"

In [ ]:
# Define Models
hand_model = mjcf.from_path(hand_path)
sawyer_model = mjcf.from_path(sawyer_path)
ball_model = mjcf.from_xml_string(ball_xml)
table_model = mjcf.from_xml_string(table_xml)


# Fingertips in XML are not actually at the tip, so we add a body with an offset
ff_tip = hand_model.find('body', 'ff_tip')
ff_tip.add('body', name='ff_tip_rubber', pos=[0, 0, 0.028])
hand_model.find('body', 'ff_tip_rubber').add('geom', type='sphere', size=[0.012], rgba=[0, 0, 0, 0])
mf_tip = hand_model.find('body', 'mf_tip')
mf_tip.add('body', name='mf_tip_rubber', pos=[0, 0, 0.028])
hand_model.find('body', 'mf_tip_rubber').add('geom', type='sphere', size=[0.012], rgba=[0, 0, 0, 0])
rf_tip = hand_model.find('body', 'rf_tip')
rf_tip.add('body', name='rf_tip_rubber', pos=[0, 0, 0.028])
hand_model.find('body', 'rf_tip_rubber').add('geom', type='sphere', size=[0.012], rgba=[0, 0, 0, 0])
th_tip = hand_model.find('body', 'th_tip')
th_tip.add('body', name='th_tip_rubber', pos=[0, 0, 0.044])
hand_model.find('body', 'th_tip_rubber').add('geom', type='sphere', size=[0.012], rgba=[0, 0, 0, 0])

In [ ]:
# Attach hand to the Sawyer
arena = mjcf.RootElement()
sawyer_site = sawyer_model.find('site', 'attachment_site')
attachment_frame = arena.attach(ball_model) # added 7 DOF to qpos for ball
arena.attach(table_model)  # table does not have DOF
sawyer_site.attach(hand_model)  # Allegro hand is a child subtree of Sawyer
arena.attach(sawyer_model)  # attach Sawyer model (7 DOF) with Allegro subtree (16 DOF)

# Set up scene
sky = arena.asset.add('texture', type='skybox', builtin="gradient", rgb1=[0, .2, 1], 
                      rgb2="1 1 1", width=512, height=512)
chequered = arena.asset.add('texture', type='2d', builtin='checker', width=500,
                            height=500, rgb1=[.2, .3, .4], rgb2=[.3, .4, .5])
grid = arena.asset.add('material', name='grid', texture=chequered,
                       texrepeat=[30, 30], reflectance=.1)
arena.worldbody.add('geom', type='plane', size=[10, 10, 10], material=grid)
for x in [-2, 2]:
  arena.worldbody.add('light', pos=[x, -1, 3], dir=[-x, 1, -2])
for y in [-2, 2]:
  arena.worldbody.add('light', pos=[-1, y, 3], dir=[1, -y, -2], attenuation=[3, 0, 0], castshadow=False)
arena.worldbody.add('camera', name='camera_1', pos=[-1, -1, 0.3], euler=[1.55, 2, 0])

# Add freejoint to ball so it can move freely
attachment_frame.add('joint', name='ball_joint', type='free', armature='5e-5')

# Extract ball elements to read attributes
ball_body = ball_model.find('body', 'ball_body')
ball_geom = ball_model.find('geom', 'ball_geom')

In [ ]:
def set_camera_position(physics, camera_name, camera_position):
    camera_id = physics.model.name2id(camera_name, "camera")
    physics.named.model.cam_pos[camera_name] = camera_position
    return camera_id

def set_initial_configuration():
    initial_qpos = [0, 0, 0, 1.0, 0, 0, 0,          # Ball (pos+quat-scalar first) (workspace represent.)
                    0, -0.8, 0, 2, 0, -1.2, 3.2,    # Sawyer arm's palm (joint space)
                    0, 0, 0, 0,                     # ff finger (joint space)
                    0, 0, 0, 0,                     # mf finger (joint space)
                    0, 0, 0, 0,                     # rf finger (joint space)
                    0, 0, 0, 0]                     # thumb (joint space)
    
    # initial_qpos = [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,]
    
    physics.data.qpos[:] = initial_qpos

physics = mjcf.Physics.from_mjcf_model(arena)

# Init scene
camera_id1 = set_camera_position(physics, "camera_1", [1.9, 0.3, 1.2])
set_initial_configuration()

physics.forward()
PIL.Image.fromarray(physics.render(height=480, width=640, camera_id=camera_id1))

In [ ]:
def set_camera_position(physics, camera_name, camera_position):
    camera_id = physics.model.name2id(camera_name, "camera")
    physics.named.model.cam_pos[camera_name] = camera_position
    return camera_id

def set_initial_configuration():
    initial_qpos = [0, 0, 0, 1, 0, 0, 0,            # Ball (pos+quat-scalar first) (workspace represent.)
                    0, -0.8, 0, 2, 0, -1.2, 3.2,    # Sawyer arm's palm (joint space)
                    0, 0, 0, 0,                     # ff finger (joint space)
                    0, 0, 0, 0,                     # mf finger (joint space)
                    0, 0, 0, 0,                     # rf finger (joint space)
                    0, 0, 0, 0]                     # thumb (joint space)
    
    # initial_qpos = [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,]
    
    physics.data.qpos[:] = initial_qpos

physics = mjcf.Physics.from_mjcf_model(arena)

# Init scene
camera_id1 = set_camera_position(physics, "camera_1", [1.9, 0.3, 1.2])
set_initial_configuration()

physics.forward()
PIL.Image.fromarray(physics.render(height=480, width=640, camera_id=camera_id1))

In [ ]:
print(physics.data.qpos) # print all the qpos
print(len(physics.data.qpos)) # print len of all the qpos
print(len(physics.data.qvel))

print()
print(physics.model.nq) # number of generalized coor. 
print(physics.model.nv)  # number of velocity
print()

# # print all the joint names
# for j in range(physics.model.njnt):
#     # notice: the use of `physics.model.ptr` the `.ptr` ensures the input is a mujoco model object
#     name = mj.mj_id2name(physics.model.ptr, mj.mjtObj.mjOBJ_JOINT, j)
#     print(j, name, physics.model.jnt_qposadr[j])

# print body id
print("palm body id is: ", physics.model.body('sawyer/allegro_right/palm').id)

# print the workspace config of a part using str id
print((physics.data.body('sawyer/allegro_right/palm').xpos))    # workspace position
print((physics.data.body('sawyer/allegro_right/palm').xquat))   # orientation in unit quaternion
print((physics.data.body('sawyer/allegro_right/palm').xmat).reshape(3,3))    # orientation in rotation matrix, q_ib
print(np.linalg.det((physics.data.body('sawyer/allegro_right/palm').xmat).reshape(3,3)))    # check deforientation in rotation matrix

# check the joint ranges
print(physics.model.jnt_range.shape)



In [ ]:
# example of computing orientation difference
# `from scipy.spatial.transform import Rotation as R``
import scipy as scp
cur_ori_quat = physics.data.body('sawyer/allegro_right/palm').xquat # q_wb

# target_ori_quat = np.array([0,0,0,1]) # q_wb
target_ori_quat = np.array([0,0,0,1]) # q_wb

diff_ori_quat = scp.spatial.transform.Rotation.from_quat(cur_ori_quat) * scp.spatial.transform.Rotation.from_quat(target_ori_quat).inv()
# diff_ori_quat = scp.spatial.transform.Rotation.from_quat(cur_ori_quat) * scp.spatial.transform.Rotation.from_quat(cur_ori_quat).inv()

# print the magnitude
print(type(diff_ori_quat))
print(diff_ori_quat.magnitude()) # [rad]



In [ ]:
target_positions = [[1.0139, -0.4455, 1.4342], [1.0566, -0.3681, 1.4689], [1.0564, -0.3731, 1.4255], [1.0765, -0.3651, 1.3784], [0.9403, -0.4003, 1.6041]]
target_orientations = [[-0.62, -0.588, -0.3918, -0.3287], [0.3301, 0.305, -0.6762, -0.5838], [0.3465, 0.2863, -0.6522, -0.6104], [0.0174, -0.0705, -0.7239, -0.686], [-0.9672, 0.061, 0.0294, 0.2446]]
target_names = ['sawyer/allegro_right/palm', 'sawyer/allegro_right/ff_tip_rubber', 'sawyer/allegro_right/mf_tip_rubber', 'sawyer/allegro_right/rf_tip_rubber', 'sawyer/allegro_right/th_tip_rubber']


print(len(target_positions))
print(len(target_orientations))
print(len(target_names))

In [ ]:
# Task 1:

## Debug flag
DEBUG_T1 = True

from multifingered_ik import LevenbergMarquardtIK

def evaluate_IK(physics, target_positions, target_orientations, target_names):
    """
    This function evaluates the IK solver for the target bodies.
    """
    model = physics.model
    data = physics.data
    step_size = 0.25
    tol = 0.002
    alpha = 0.5
    n = len(target_positions)
    jacp = np.zeros((n, 3, model.nv)) # translational Jacobian
    jacr = np.zeros((n, 3, model.nv)) # rotational Jacobian
    damping = 0.15
    max_steps = 200
    # max_steps = 10     # [MT]

    ik = LevenbergMarquardtIK(model, data, step_size, tol, alpha, jacp, jacr, damping, max_steps, physics)
    final_qpos = ik.calculate(target_positions, target_orientations, target_names)
    return final_qpos

# YOUR CODE HERE: Fill these in from lab doc
target_positions = [[1.0139, -0.4455, 1.4342], [1.0566, -0.3681, 1.4689], [1.0564, -0.3731, 1.4255], [1.0765, -0.3651, 1.3784], [0.9403, -0.4003, 1.6041]]
target_orientations = [[-0.62, -0.588, -0.3918, -0.3287], [0.3301, 0.305, -0.6762, -0.5838], [0.3465, 0.2863, -0.6522, -0.6104], [0.0174, -0.0705, -0.7239, -0.686], [-0.9672, 0.061, 0.0294, 0.2446]]
target_names = ['sawyer/allegro_right/palm', 'sawyer/allegro_right/ff_tip_rubber', 'sawyer/allegro_right/mf_tip_rubber', 'sawyer/allegro_right/rf_tip_rubber', 'sawyer/allegro_right/th_tip_rubber']

physics.reset() # 
final_qpos = evaluate_IK(physics, target_positions, target_orientations, target_names)


if DEBUG_T1:
    print(final_qpos) 

physics.data.qpos[:] = final_qpos
physics.forward()
PIL.Image.fromarray(physics.render(height=480, width=640, camera_id=camera_id1))

In [ ]:
# check desired position
target_pos = ball_body.pos.copy()
target_pos[2] += 0.1
print(ball_body.pos)
print(target_pos)

## Video Creation

In [ ]:
destination_config = final_qpos.copy()

In [ ]:
initial_qpos = [0, 0, 0, 1, 0, 0, 0,            # Ball (pos+quat-scalar first) (workspace represent.)
                0, -0.8, 0, 2, 0, -1.2, 3.2,    # Sawyer arm's palm (joint space)
                0, 0, 0, 0,                     # ff finger (joint space)
                0, 0, 0, 0,                     # mf finger (joint space)
                0, 0, 0, 0,                     # rf finger (joint space)
                0, 0, 0, 0]                     # thumb (joint space)

initial_config = np.array(initial_qpos)

In [ ]:
print(destination_config)
print(destination_config.shape)
print(final_qpos)

print()

print(initial_config)
print(initial_config.shape)

In [ ]:
physics.reset()
set_initial_configuration()
physics.data.qpos[:] = initial_config
physics.forward()
start_img = PIL.Image.fromarray(physics.render(height=480, width=640, camera_id=camera_id1))

physics.data.qpos[:] = destination_config
physics.forward()
end_img = PIL.Image.fromarray(physics.render(height=480, width=640, camera_id=camera_id1))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(np.asarray(start_img))
ax[0].set_title("Initial Configuration")
ax[0].axis("off")

ax[1].imshow(np.asarray(end_img))
ax[1].set_title("Final IK Configuration")
ax[1].axis("off")

plt.tight_layout()

In [ ]:
duration = 4.0
framerate = 30
n_steps = int(duration * framerate)

video = []

def smoothstep(t):
    return t * t * (3.0 - 2.0 * t)

with mj.Renderer(physics.model.ptr, 480, 640) as renderer:
    for i in range(n_steps):
        tau = i / max(n_steps - 1, 1)
        tau = smoothstep(tau)

        q_interp = (1.0 - tau) * initial_config + tau * destination_config

        physics.data.qpos[:] = q_interp
        physics.forward()

        renderer.update_scene(physics.data.ptr, "camera_1")
        frame = np.array(renderer.render()).copy()
        video.append(frame)

display_video(video, framerate=framerate)

In [ ]:
import mediapy as media

video_path = "task1_ik_motion.mp4"
media.write_video(video_path, video, fps=framerate)
print(f"Saved video to {video_path}")